# Monitor HCB - Split por Regional

Este notebook escanea todos los archivos generados en `REGIONALES/` y produce una tabla consolidada con métricas para Power BI.

**Qué hace:**
1. Lee cada archivo `MATRIZ_{REGIONAL}.xlsx`
2. Extrae métricas: filas, contratos, cupos, valores, alertas, errores, vacíos
3. Calcula campos derivados (promedios, porcentajes, semáforo de estado)
4. Genera un Excel con dos hojas: DATOS + DICCIONARIO

**Frecuencia sugerida:** ejecutar cada vez que los usuarios devuelvan sus archivos modificados, antes de refrescar Power BI.

In [1]:
import os, re, time, unicodedata
from datetime import datetime
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

## 1. Configuración

In [2]:
REGIONALES_DIR = Path(r"d:\ICBF\cost-tracking\data\replicacion hcb 5 junio\REGIONALES")
OUTPUT_DIR = Path(r"d:\ICBF\cost-tracking\data\monitoring")
OUTPUT_FILE = OUTPUT_DIR / "monitor_hcb.xlsx"
SHEET_NAME = "MATRIZ"

SECOP_FILE = REGIONALES_DIR.parent / "Contratos_ICBF_SECOP_II_20260324.xlsx"

VALOR_COMPARAR_SECOP = "VALOR TOTAL INICIAL DEL CONTRATO (UNICO REGISTRO"  # ← cámbialo aquí. Alternativas: "VALOR ACTUAL DEL CONTRATO ANTES DE LA ADICI", "VALOR TOTAL INICIAL APORTE ICBF", "VALOR TOTAL INICIAL DEL CONTRATO POR SERVICIO"
TOLERANCIA_SECOP = 1  # diferencia máxima en pesos para considerar valores iguales

KEYWORDS = {
    "REGIONAL":   ["REGIONAL"],
    "REF_CONTRATO": ["REFERENCIA", "CONTRATO SECOP"],
    "RP":         ["No. RP"],
    "MUNICIPIO":  ["MUNICIPIO"],
    "SERVICIO":   ["NOMBRE DEL SERVICIO"],
    "CUPOS":      ["SUMATORIA DE LOS CUPOS"],
    "EAS":        ["NOMBRE EAS"],
    "NIT":        ["NIT"],
    "VALOR_INI":  ["VALOR TOTAL INICIAL APORTE ICBF"],
    "VALOR_ADC":  ["VALOR ADICION NIVELACION CANASTA (UNICO"],
    "VALOR_SECOP_COMP": [VALOR_COMPARAR_SECOP],
    "ALERTA_1000":["ALERTA DE AVAL", "1000 SMLMV"],
    "ALERTA_5000":["ALERTA CONTRATO DE APORTE", "5000 SMLMV"],
}

DICT = {
    "REGIONAL": "Nombre de la regional",
    "ARCHIVO": "Ruta del archivo MATRIZ",
    "FECHA_SCAN": "Momento del escaneo",
    "TAMANO_KB": "Tamano del archivo en KB",
    "ULTIMA_MODIFICACION": "Ultima modificacion del archivo",
    "DIAS_DESDE_MOD": "Dias desde la ultima modificacion",
    "FILAS_DATOS": "Filas con datos en la hoja MATRIZ",
    "COLUMNAS": "Columnas en la hoja MATRIZ",
    "HOJAS": "Hojas del libro",
    "REGIONAL_EN_DATOS": "Regional en la primera fila de datos",
    "ERRORES_REF": "Celdas con error #REF!",
    "VACIOS_CRITICOS": "Celdas vacias en columnas clave",
    "PCT_VACIOS": "% celdas criticas vacias",
    "TIENE_ERRORES": "SI = tiene #REF!",
    "TIENE_VACIOS": "SI = tiene vacios criticos",
    "TIENE_ALERTAS": "SI = tiene alertas",
    "CONTRATOS_UNICOS": "Contratos distintos",
    "MUNICIPIOS": "Municipios distintos",
    "EAS_UNICOS": "Contratistas distintos",
    "SERVICIOS": "Tipos de servicio",
    "CUPOS_TOTALES": "Suma total de cupos",
    "CUPOS_X_CONTRATO": "Promedio cupos por contrato",
    "VALOR_INICIAL_TOTAL": "Valor inicial total",
    "VALOR_INICIAL_X_CONTRATO": "Valor inicial promedio",
    "VALOR_ADICIONAR_TOTAL": "Valor adicion total",
    "VALOR_ADICION_X_CONTRATO": "Valor adicion promedio",
    "ALERTA_1000": "Alertas >1000 SMLMV",
    "ALERTA_5000": "Alertas >5000 SMLMV",
    "CONTRATOS_NO_ENCONTRADOS_SECOP": "Contratos unicos no encontrados en la base SECOP",
    "CONTRATOS_VALOR_DIFERENTE_SECOP": "Contratos unicos cuyo valor difiere del reportado en SECOP",
    "ESTADO": "VERDE=OK / AMARILLO=vacios / NARANJA=alertas / ROJO=errores",
}

## 2. Funciones auxiliares

In [3]:
def _strip_accents(text):
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def _match_col(headers, keywords):
    for col_idx, hdr in sorted(headers.items()):
        hdr_clean = _strip_accents(hdr.split(chr(10))[0]).strip().upper()
        for kw in keywords:
            if _strip_accents(kw).upper() in hdr_clean:
                return col_idx
    return None

## 2b. Cargar base SECOP

In [ ]:
def _parse_secop_valor(val):
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        return float(val.replace("$", "").replace(",", "").strip())
    return 0.0


print(f"Cargando SECOP desde: {SECOP_FILE}")
wb_secop = load_workbook(SECOP_FILE, data_only=True, read_only=True)
ws_secop = wb_secop.active
secop_headers = [cell.value for cell in next(ws_secop.iter_rows(max_row=1))]

ref_col = val_col = None
for i, h in enumerate(secop_headers):
    if h and "Referencia del Contrato" in str(h):
        ref_col = i
    if h and "Valor del Contrato" in str(h):
        val_col = i
        break

secop_dict = {}
for row in ws_secop.iter_rows(min_row=2, values_only=True):
    ref = row[ref_col]
    val = row[val_col]
    if ref is not None:
        secop_dict[str(ref).strip()] = _parse_secop_valor(val)

wb_secop.close()
print(f"SECOP cargado: {len(secop_dict)} contratos")

## 3. Escaneo de un archivo

In [4]:
def scan_file(filepath, folder_name, secop_dict):
    stats = filepath.stat()
    metrics = {
        "REGIONAL": folder_name,
        "ARCHIVO": str(filepath.resolve()),
        "FECHA_SCAN": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "TAMANO_KB": round(stats.st_size / 1024, 1),
        "ULTIMA_MODIFICACION": datetime.fromtimestamp(stats.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
    }

    wb = load_workbook(filepath, data_only=True, read_only=True)
    ws = wb[SHEET_NAME]
    metrics["COLUMNAS"] = ws.max_column or 70
    metrics["HOJAS"] = "|".join(wb.sheetnames)

    headers = {}
    for row in ws.iter_rows(min_row=2, max_row=2, values_only=True):
        for ci, val in enumerate(row, start=1):
            if val:
                headers[ci] = str(val).strip()

    col_map = {k: _match_col(headers, kw) for k, kw in KEYWORDS.items()}

    data_rows = errors_ref = empty_critical = 0
    total_cupos = total_valor_ini = total_valor_adc = 0.0
    contracts, municipalities, eas_set, services = set(), set(), set(), set()
    alerta_1000 = alerta_5000 = 0
    contratos_no_encontrados = contratos_valor_diferente = 0
    regional_en_datos = ""
    crit_cols = [col_map[k] for k in ["REF_CONTRATO","MUNICIPIO","EAS","RP","SERVICIO"] if col_map.get(k)]
    first = True
    contract_details = {}

    for row in ws.iter_rows(min_row=3, values_only=True):
        rd = {i+1: v for i, v in enumerate(row)}
        if all(v is None for v in rd.values()):
            continue
        data_rows += 1
        for v in rd.values():
            if isinstance(v, str) and "#REF!" in v:
                errors_ref += 1
        c = col_map
        if first and c.get("REGIONAL") and rd.get(c["REGIONAL"]):
            regional_en_datos = str(rd[c["REGIONAL"]]).strip()
            first = False
        if c.get("REF_CONTRATO") and rd.get(c["REF_CONTRATO"]):
            v = rd[c["REF_CONTRATO"]]
            contracts.add(str(int(v)) if isinstance(v, (int, float)) else str(v).strip())
        if c.get("MUNICIPIO") and rd.get(c["MUNICIPIO"]):
            municipalities.add(str(rd[c["MUNICIPIO"]]).strip().upper())
        if c.get("EAS") and rd.get(c["EAS"]):
            eas_set.add(str(rd[c["EAS"]]).strip().upper())
        if c.get("SERVICIO") and rd.get(c["SERVICIO"]):
            services.add(str(rd[c["SERVICIO"]]).strip())
        if c.get("CUPOS") and isinstance(rd.get(c["CUPOS"]), (int, float)):
            total_cupos += rd[c["CUPOS"]]
        if c.get("VALOR_INI") and isinstance(rd.get(c["VALOR_INI"]), (int, float)):
            total_valor_ini += rd[c["VALOR_INI"]]
        if c.get("VALOR_ADC") and isinstance(rd.get(c["VALOR_ADC"]), (int, float)):
            total_valor_adc += rd[c["VALOR_ADC"]]

        ref_col_idx = c.get("REF_CONTRATO")
        val_col_idx = c.get("VALOR_SECOP_COMP")
        if ref_col_idx and val_col_idx:
            ref_raw = rd.get(ref_col_idx)
            val_raw = rd.get(val_col_idx)
            if ref_raw is not None and isinstance(val_raw, (int, float)) and val_raw > 0:
                ref_str = str(int(ref_raw)) if isinstance(ref_raw, (int, float)) else str(ref_raw).strip()
                if ref_str not in contract_details:
                    secop_val = secop_dict.get(ref_str)
                    if secop_val is None:
                        alerta = "NO_ENCONTRADO"
                        contratos_no_encontrados += 1
                    else:
                        diff = abs(secop_val - float(val_raw))
                        if diff > TOLERANCIA_SECOP:
                            alerta = "VALOR_DIFERENTE"
                            contratos_valor_diferente += 1
                        else:
                            alerta = "OK"
                    contract_details[ref_str] = {
                        "REGIONAL": folder_name,
                        "REF_CONTRATO": ref_str,
                        "VALOR_MATRIZ": float(val_raw),
                        "VALOR_SECOP": secop_val,  # None si no encontrado en SECOP
                        "ALERTA": alerta,
                    }

        if c.get("ALERTA_1000") and rd.get(c["ALERTA_1000"]):
            v = str(rd[c["ALERTA_1000"]]).strip().upper()
            if v in ("SI","REQUIERE AVAL","ALERTA","ALERTA MAYOR QUE 1.000"):
                alerta_1000 += 1
        if c.get("ALERTA_5000") and rd.get(c["ALERTA_5000"]):
            v = str(rd[c["ALERTA_5000"]]).strip().upper()
            if v in ("SI","REQUIERE AVAL","ALERTA","ALERTA MAYOR QUE 5.000"):
                alerta_5000 += 1
        for cc in crit_cols:
            if rd.get(cc) is None:
                empty_critical += 1

    wb.close()
    metrics.update(dict(
        REGIONAL_EN_DATOS=regional_en_datos, FILAS_DATOS=data_rows,
        ERRORES_REF=errors_ref, VACIOS_CRITICOS=empty_critical,
        CONTRATOS_UNICOS=len(contracts), MUNICIPIOS=len(municipalities),
        EAS_UNICOS=len(eas_set),
        SERVICIOS="|".join(sorted(services)) if services else "",
        CUPOS_TOTALES=total_cupos, VALOR_INICIAL_TOTAL=total_valor_ini,
        VALOR_ADICIONAR_TOTAL=total_valor_adc,
        ALERTA_1000=alerta_1000, ALERTA_5000=alerta_5000,
        CONTRATOS_NO_ENCONTRADOS_SECOP=contratos_no_encontrados,
        CONTRATOS_VALOR_DIFERENTE_SECOP=contratos_valor_diferente,
    ))
    return metrics, contract_details

## 4. Campos calculados y semáforo

In [5]:
def add_calculated_fields(df):
    expected = df["FILAS_DATOS"] * 5
    df["PCT_VACIOS"] = (df["VACIOS_CRITICOS"] / expected.replace(0, 1) * 100).round(1)
    df["TIENE_ERRORES"] = df["ERRORES_REF"].apply(lambda x: "SI" if x > 0 else "NO")
    df["TIENE_VACIOS"] = df["VACIOS_CRITICOS"].apply(lambda x: "SI" if x > 0 else "NO")
    df["TIENE_ALERTAS"] = df.apply(
        lambda r: "SI" if r["ALERTA_1000"] + r["ALERTA_5000"] > 0 else "NO", axis=1
    )
    nz = df["CONTRATOS_UNICOS"].replace(0, 1)
    df["CUPOS_X_CONTRATO"] = (df["CUPOS_TOTALES"] / nz).round(0).astype("Int64")
    df["VALOR_INICIAL_X_CONTRATO"] = (df["VALOR_INICIAL_TOTAL"] / nz).round(0).astype("Int64")
    df["VALOR_ADICION_X_CONTRATO"] = (df["VALOR_ADICIONAR_TOTAL"] / nz).round(0).astype("Int64")
    now = datetime.now()
    df["DIAS_DESDE_MOD"] = df["ULTIMA_MODIFICACION"].apply(
        lambda x: (now - datetime.strptime(x, "%Y-%m-%d %H:%M:%S")).days
        if isinstance(x, str) else 0
    )

    def estado(r):
        if r["ERRORES_REF"] > 0:
            return "ROJO"
        if r["ALERTA_1000"] + r["ALERTA_5000"] > 0:
            return "NARANJA"
        if r["VACIOS_CRITICOS"] > 0:
            return "AMARILLO"
        return "VERDE"
    df["ESTADO"] = df.apply(estado, axis=1)
    return df

## 5. Ejecutar escaneo

In [6]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
rows = []
all_contract_details = []
for d in sorted(REGIONALES_DIR.iterdir()):
    if not d.is_dir() or d.name.startswith("_"):
        continue
    fp = d / f"MATRIZ_{d.name}.xlsx"
    if not fp.exists():
        continue
    print(f"[{len(rows)+1}/?] {d.name}...", end=" ", flush=True)
    t0 = time.time()
    try:
        metrics, contract_details = scan_file(fp, d.name, secop_dict)
        rows.append(metrics)
        all_contract_details.extend(contract_details.values())
        print(f"OK ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"ERROR: {e}")

df = add_calculated_fields(pd.DataFrame(rows))
df_detail = pd.DataFrame(all_contract_details) if all_contract_details else pd.DataFrame(columns=["REGIONAL","REF_CONTRATO","VALOR_MATRIZ","VALOR_SECOP","ALERTA"])
print(f"\nEscaneados: {len(df)} archivos, {df['FILAS_DATOS'].sum():,} filas totales")
print(f"Detalle SECOP: {len(df_detail)} contratos evaluados")

[1/?] ANTIOQUIA... OK (1.4s)
[2/?] ARAUCA... 

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


OK (1.4s)
[3/?] ATLANTICO... OK (1.7s)
[4/?] BOGOTA_DC... OK (1.8s)
[5/?] BOLIVAR... OK (1.8s)
[6/?] BOYACA... OK (1.8s)
[7/?] CALDAS... OK (1.8s)
[8/?] CAQUETA... OK (1.8s)
[9/?] CAUCA... OK (1.8s)
[10/?] CESAR... OK (1.8s)
[11/?] CHOCO... OK (1.8s)
[12/?] CORDOBA... OK (1.9s)
[13/?] CUNDINAMARCA... OK (1.8s)
[14/?] GUAVIARE... OK (1.8s)
[15/?] HUILA... OK (1.8s)
[16/?] LA_GUAJIRA... OK (1.8s)
[17/?] MAGDALENA... OK (1.9s)
[18/?] META... OK (1.9s)
[19/?] NARINO... OK (1.8s)
[20/?] NORTE_DE_SANTANDER... OK (1.8s)
[21/?] PUTUMAYO... OK (1.8s)
[22/?] QUINDIO... OK (1.8s)
[23/?] RISARALDA... OK (1.8s)
[24/?] SAN_ANDRES... OK (1.9s)
[25/?] SANTANDER... OK (1.9s)
[26/?] SUCRE... OK (1.9s)
[27/?] TOLIMA... OK (1.8s)
[28/?] VALLE_DEL_CAUCA... OK (1.9s)
[29/?] VICHADA... OK (1.8s)

Escaneados: 29 archivos, 3,098 filas totales


## 6. Tabla consolidada

In [7]:
cols_show = [
    "REGIONAL","ESTADO","FILAS_DATOS","CONTRATOS_UNICOS",
    "CUPOS_TOTALES","VALOR_INICIAL_TOTAL","VALOR_ADICIONAR_TOTAL",
    "ERRORES_REF","PCT_VACIOS","ALERTA_1000","ALERTA_5000","DIAS_DESDE_MOD",
]
df[cols_show]

,REGIONAL,ESTADO,FILAS_DATOS,CONTRATOS_UNICOS,CUPOS_TOTALES,VALOR_INICIAL_TOTAL,VALOR_ADICIONAR_TOTAL,ERRORES_REF,PCT_VACIOS,ALERTA_1000,ALERTA_5000,DIAS_DESDE_MOD
0,ANTIOQUIA,NARANJA,244,107,45264.0,1.865799e+11,6.779870e+10,0,0.0,211,105,0
1,ARAUCA,VERDE,4,3,1158.0,0.000000e+00,1.466792e+09,0,0.0,0,0,0
2,ATLANTICO,NARANJA,149,73,48334.0,1.508418e+09,7.409177e+10,0,0.0,44,0,0
3,BOGOTA_DC,NARANJA,406,205,41052.0,3.052627e+10,5.842333e+10,0,0.0,17,0,0
4,BOLIVAR,NARANJA,204,108,44093.0,1.512887e+09,6.880981e+10,0,19.6,31,0,0
5,BOYACA,NARANJA,253,61,40961.0,0.000000e+00,6.289775e+10,0,2.0,107,0,0
6,CALDAS,NARANJA,17,4,5351.0,0.000000e+00,6.937681e+09,0,0.0,12,0,0
7,CAQUETA,VERDE,9,5,1638.0,0.000000e+00,1.943057e+09,0,0.0,0,0,0
8,CAUCA,NARANJA,132,70,27460.0,0.000000e+00,4.202341e+10,0,18.9,15,0,0
9,CESAR,NARANJA,73,31,32920.0,0.000000e+00,5.180203e+10,0,0.0,39,0,0


## 7. Resumen ejecutivo

In [8]:
print("=" * 60)
print(f"  TOTAL ARCHIVOS:         {len(df)}")
print(f"  FILAS TOTALES:          {df['FILAS_DATOS'].sum():,}")
print(f"  CONTRATOS UNICOS:       {df['CONTRATOS_UNICOS'].sum():,}")
print(f"  CUPOS TOTALES:          {df['CUPOS_TOTALES'].sum():,.0f}")
print(f"  VALOR INICIAL TOTAL:    ${df['VALOR_INICIAL_TOTAL'].sum():,.0f}")
print(f"  VALOR ADICION TOTAL:    ${df['VALOR_ADICIONAR_TOTAL'].sum():,.0f}")
print("=" * 60)
print(f"\n  DISTRIBUCION POR ESTADO:")
for k, v in df["ESTADO"].value_counts().items():
    print(f"    {k}: {v}")
print(f"\n  ARCHIVOS CON ERRORES:   {len(df[df['ERRORES_REF'] > 0])}")
print(f"  ARCHIVOS CON VACIOS:    {len(df[df['VACIOS_CRITICOS'] > 0])}")
print(f"  ARCHIVOS CON ALERTAS:   {len(df[df['ALERTA_1000'] + df['ALERTA_5000'] > 0])}")
print("=" * 60)

  TOTAL ARCHIVOS:         29
  FILAS TOTALES:          3,098
  CONTRATOS UNICOS:       1,464
  CUPOS TOTALES:          654,407
  VALOR INICIAL TOTAL:    $220,127,518,279
  VALOR ADICION TOTAL:    $997,594,069,080

  DISTRIBUCION POR ESTADO:
    NARANJA: 22
    VERDE: 7

  ARCHIVOS CON ERRORES:   0
  ARCHIVOS CON VACIOS:    8
  ARCHIVOS CON ALERTAS:   22


## 8. Generar Excel para Power BI

El archivo se genera con dos hojas:
- **DATOS**: La tabla plana con todas las métricas (29 columnas)
- **DICCIONARIO**: Descripción de cada columna

En Power BI solo conectas este Excel y arrastras los campos a tus visuales.

In [9]:
COLUMNAS_FINAL = [
    "REGIONAL","ARCHIVO","FECHA_SCAN","TAMANO_KB","ULTIMA_MODIFICACION","DIAS_DESDE_MOD",
    "FILAS_DATOS","COLUMNAS","HOJAS","REGIONAL_EN_DATOS",
    "ERRORES_REF","VACIOS_CRITICOS","PCT_VACIOS","TIENE_ERRORES","TIENE_VACIOS",
    "CONTRATOS_UNICOS","MUNICIPIOS","EAS_UNICOS","SERVICIOS",
    "CUPOS_TOTALES","CUPOS_X_CONTRATO","VALOR_INICIAL_TOTAL","VALOR_INICIAL_X_CONTRATO",
    "VALOR_ADICIONAR_TOTAL","VALOR_ADICION_X_CONTRATO",
    "ALERTA_1000","ALERTA_5000","TIENE_ALERTAS",
    "CONTRATOS_NO_ENCONTRADOS_SECOP","CONTRATOS_VALOR_DIFERENTE_SECOP",
    "ESTADO",
]
df_out = df[[c for c in COLUMNAS_FINAL if c in df.columns]]

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df_out.to_excel(writer, sheet_name="DATOS", index=False)
    dict_df = pd.DataFrame([
        {"COLUMNA": k, "DESCRIPCION": v} for k, v in DICT.items()
    ])
    dict_df.to_excel(writer, sheet_name="DICCIONARIO", index=False)
    if not df_detail.empty:
        df_detail.to_excel(writer, sheet_name="DETALLE_ALERTAS_SECOP", index=False)

print(f"Excel generado: {OUTPUT_FILE}")
print(f"  DATOS: {len(df_out)} filas x {len(df_out.columns)} columnas")
print(f"  DICCIONARIO: {len(dict_df)} columnas documentadas")
if not df_detail.empty:
    print(f"  DETALLE_ALERTAS_SECOP: {len(df_detail)} contratos evaluados")

Excel generado: d:\ICBF\cost-tracking\data\monitoring\monitor_hcb.xlsx
  DATOS: 29 filas x 29 columnas
  DICCIONARIO: 29 columnas documentadas


---
**Nota:** Para refrescar los datos cuando los usuarios modifiquen sus archivos, solo ejecuta este notebook nuevamente (Kernel > Restart & Run All) y luego refresca el origen en Power BI.